# Sampling

With a finite number of shots, MatchCake draws samples of a matchgate circuit $V$ applied
to a qubit product state $|\psi_\text{prod}\rangle = \bigotimes_{k=0}^{n-1}(a_k|0\rangle + b_k|1\rangle)$.
The samples are used by `qml.sample`, and by `qml.expval` and `qml.probs` when shots are set.

Samples are not limited to the computational basis. Each qubit can be measured in its own
single-qubit basis, such as the $X$ or $Y$ basis, even though the rotation to that basis
(e.g. a Hadamard) is not a matchgate. Brod showed that such measurements can still be sampled
efficiently [[1]](https://arxiv.org/abs/1602.03539). MatchCake measures the qubits one at a time
on the evolved Majorana covariance matrix, so a sample costs $O(n^3)$, in any product basis.

In [1]:
import time

import numpy as np
import pennylane as qml
import torch

import matchcake as mc
from matchcake.devices.sampling_strategies import ProductBasisSampling
from matchcake.operations.state_preparation import ProductState

# The samples are drawn from the torch random generator.
_ = torch.manual_seed(0)

## Matchgate circuit

We use a random single-particle transition matrix (SPTM) on 4 qubits as the matchgate
circuit $V$, applied to the basis state $|1010\rangle$. The equivalent qubit unitary `U` is
used only to cross-check against `default.qubit`.

In [2]:
n = 4
wires = list(range(n))
n_shots = 20_000
state_prep = ProductState.from_basis_state([1, 0, 1, 0], wires=wires)
sptm = mc.operations.SingleParticleTransitionMatrixOperation.random(wires=wires, seed=0)
U = sptm.to_qubit_operation()


def bits_to_indices(bits):
    # (shots, n) bits -> (shots,) basis state indices, the first wire being the most significant bit.
    return np.asarray(bits, dtype=int) @ (1 << np.arange(bits.shape[-1])[::-1])


def frequencies(bits):
    return np.bincount(bits_to_indices(bits), minlength=2 ** bits.shape[-1]) / len(bits)

## Computational basis samples

Shots are given with `qml.set_shots` (or with `shots=` when the device is built).
`qml.sample()` returns one row of bits per shot. Their frequencies match the exact
probabilities of `qml.probs` up to statistical fluctuations of order $1/\sqrt{\text{shots}}$.

In [3]:
@qml.set_shots(n_shots)
@qml.qnode(mc.NIFDevice())
def sample_circuit():
    state_prep.queue()
    sptm.queue()
    return qml.sample()


@qml.qnode(qml.device("default.qubit"))
def svs_probs_circuit():
    state_prep.queue()
    U.queue()
    return qml.probs(wires=wires)


samples = np.asarray(sample_circuit())
empirical = frequencies(samples)
exact = np.asarray(svs_probs_circuit())
print(f"samples shape = {samples.shape}")
print(f"first samples =\n{samples[:3]}")
print(f"empirical     = {np.round(empirical, 3)}")
print(f"exact         = {np.round(exact, 3)}")
np.testing.assert_allclose(empirical, exact, atol=2e-2)
print("Frequencies agree with state-vector simulation.")

samples shape = (20000, 4)
first samples =
[[1 1 1 1]
 [1 0 1 0]
 [0 1 1 0]]
empirical     = [0.03  0.    0.    0.342 0.    0.081 0.112 0.    0.    0.051 0.089 0.
 0.022 0.    0.    0.274]
exact         = [0.03  0.    0.    0.343 0.    0.081 0.111 0.    0.    0.052 0.09  0.
 0.022 0.    0.    0.272]
Frequencies agree with state-vector simulation.


## Expectation values from shots

With shots, `qml.expval` is estimated from the samples instead of being computed exactly.

In [4]:
observable = qml.Z(0) @ qml.Z(1)


@qml.set_shots(n_shots)
@qml.qnode(mc.NIFDevice())
def expval_circuit():
    state_prep.queue()
    sptm.queue()
    return qml.expval(observable)


@qml.qnode(qml.device("default.qubit"))
def svs_expval_circuit():
    state_prep.queue()
    U.queue()
    return qml.expval(observable)


estimate = float(expval_circuit())
exact_expval = float(svs_expval_circuit())
print(f"<Z0 Z1> estimate = {estimate:.4f}, exact = {exact_expval:.4f}")
np.testing.assert_allclose(estimate, exact_expval, atol=2e-2)
print("Results agree with state-vector simulation.")

<Z0 Z1> estimate = 0.3277, exact = 0.3319
Results agree with state-vector simulation.


## Sampling in the X basis

Measuring a qubit in the $X$ basis amounts to applying a Hadamard before a computational basis
measurement. The Hadamard is not a matchgate, so it cannot be part of the circuit on the NIF
device. Instead, it is given by the observable: `qml.sample(qml.X(k))` returns the eigenvalue
$\pm 1$ of $X_k$ for each shot.

Measuring every wire in the $X$ basis in the same circuit gives the joint $X$-basis outcomes.
Bit $0$ stands for the eigenvalue $+1$ and bit $1$ for $-1$, so the frequencies match the
computational basis probabilities of `default.qubit` with a final layer of Hadamards.

In [5]:
@qml.set_shots(n_shots)
@qml.qnode(mc.NIFDevice())
def x_sample_circuit():
    state_prep.queue()
    sptm.queue()
    return [qml.sample(qml.X(wire)) for wire in wires]


@qml.qnode(qml.device("default.qubit"))
def svs_hadamard_layer_circuit():
    state_prep.queue()
    U.queue()
    for wire in wires:
        qml.Hadamard(wires=wire)
    return qml.probs(wires=wires)


x_eigenvalues = np.stack([np.asarray(result) for result in x_sample_circuit()], axis=-1)  # (shots, n)
x_bits = (1 - x_eigenvalues) // 2
x_empirical = frequencies(x_bits.astype(int))
x_exact = np.asarray(svs_hadamard_layer_circuit())
print(f"empirical = {np.round(x_empirical, 3)}")
print(f"exact     = {np.round(x_exact, 3)}")
np.testing.assert_allclose(x_empirical, x_exact, atol=2e-2)
print("Frequencies agree with state-vector simulation.")

empirical = [0.069 0.068 0.013 0.071 0.107 0.012 0.084 0.072 0.074 0.083 0.013 0.11
 0.072 0.012 0.067 0.072]
exact     = [0.067 0.07  0.012 0.073 0.111 0.012 0.083 0.073 0.073 0.083 0.012 0.111
 0.073 0.012 0.07  0.067]
Frequencies agree with state-vector simulation.


## Pauli products and Hamiltonians

Any product of single-qubit Paulis can be sampled, as long as each wire is measured in a single
basis within a circuit. A Hamiltonian is split by PennyLane into groups of terms that commute
qubit-wise, and each group is estimated from its own shots.

In [6]:
hamiltonian = qml.Hamiltonian(
    [0.5, -1.2, 0.8, 0.3],
    [qml.X(0) @ qml.X(1), qml.Y(1) @ qml.Y(2), qml.Z(0) @ qml.Z(3), qml.X(2) @ qml.Y(3)],
)


@qml.set_shots(n_shots)
@qml.qnode(mc.NIFDevice())
def hamiltonian_circuit():
    state_prep.queue()
    sptm.queue()
    return qml.expval(hamiltonian)


@qml.qnode(qml.device("default.qubit"))
def svs_hamiltonian_circuit():
    state_prep.queue()
    U.queue()
    return qml.expval(hamiltonian)


estimate = float(hamiltonian_circuit())
exact_expval = float(svs_hamiltonian_circuit())
print(f"<H> estimate = {estimate:.4f}, exact = {exact_expval:.4f}")
np.testing.assert_allclose(estimate, exact_expval, atol=5e-2)
print("Results agree with state-vector simulation.")

<H> estimate = -0.7076, exact = -0.7103
Results agree with state-vector simulation.


## Arbitrary product bases

The device method `generate_samples` takes the basis of each wire explicitly, either as a
Pauli letter or as a Bloch vector $\hat{n}$. A wire with Bloch vector $\hat{n}$ is measured
along $\hat{n} \cdot \vec{\sigma}$, with bit $0$ for the eigenvalue $+1$. Wires that are not
listed are measured in the computational basis.

Here, wire 0 is measured in the $Y$ basis and wire 1 along $\hat{n} = (\sin\theta, 0, \cos\theta)$,
which is the same as applying $R_Y(-\theta)$ before a computational basis measurement. The
operations are applied to the device directly with `apply`.

In [7]:
theta = np.pi / 3
device = mc.NIFDevice(wires=n, shots=n_shots)
device.apply([state_prep, sptm])
bases = {0: "Y", 1: [np.sin(theta), 0.0, np.cos(theta)]}
basis_samples = device.generate_samples(bases=bases)


@qml.qnode(qml.device("default.qubit"))
def svs_rotated_circuit():
    state_prep.queue()
    U.queue()
    qml.adjoint(qml.S(wires=0))
    qml.Hadamard(wires=0)
    qml.RY(-theta, wires=1)
    return qml.probs(wires=wires)


basis_empirical = frequencies(basis_samples)
basis_exact = np.asarray(svs_rotated_circuit())
print(f"empirical = {np.round(basis_empirical, 3)}")
print(f"exact     = {np.round(basis_exact, 3)}")
np.testing.assert_allclose(basis_empirical, basis_exact, atol=2e-2)
print("Frequencies agree with state-vector simulation.")

empirical = [0.003 0.018 0.011 0.053 0.023 0.049 0.091 0.251 0.025 0.04  0.08  0.274
 0.001 0.025 0.017 0.039]
exact     = [0.003 0.019 0.011 0.053 0.023 0.048 0.089 0.254 0.025 0.041 0.084 0.272
 0.001 0.026 0.016 0.035]
Frequencies agree with state-vector simulation.


## Batched circuits

Parameter broadcasting works with shots too: a batch of parameters gives one estimate per
batch entry, and `qml.sample` returns samples of shape `(batch, shots, ...)`, as on
`default.qubit`.

In [8]:
angles = np.array([0.1, 0.8, 2.0])


def batched_circuit(theta):
    qml.BasisState(np.array([0, 1]), wires=[0, 1])
    qml.SingleExcitation(theta, wires=[0, 1])
    return qml.expval(qml.X(0) @ qml.X(1))


batched_estimate = qml.set_shots(qml.QNode(batched_circuit, mc.NIFDevice()), n_shots)(angles)
batched_exact = qml.QNode(batched_circuit, qml.device("default.qubit"))(angles)
print(f"estimates = {np.round(np.asarray(batched_estimate, dtype=float), 3)}")
print(f"exact     = {np.round(np.asarray(batched_exact, dtype=float), 3)}")
np.testing.assert_allclose(np.asarray(batched_estimate, dtype=float), batched_exact, atol=2e-2)
print("Results agree with state-vector simulation.")

estimates = [0.099 0.72  0.91 ]
exact     = [0.1   0.717 0.909]
Results agree with state-vector simulation.


## Scaling to many qubits

A sample costs $O(n^3)$ and the samples are drawn in vectorized chunks of shots, so circuits far
beyond the reach of state-vector simulation can be sampled. Below, a random matchgate circuit on
50 qubits is sampled in the $X$ basis on every wire.

In [9]:
n_large = 50
large_wires = list(range(n_large))


@qml.set_shots(1_000)
@qml.qnode(mc.NIFDevice())
def large_circuit():
    ProductState.from_basis_state([1, 0] * (n_large // 2), wires=large_wires)
    mc.operations.SingleParticleTransitionMatrixOperation.random(wires=large_wires, seed=1)
    return qml.expval(qml.X(0) @ qml.X(1))


start = time.perf_counter()
large_estimate = float(large_circuit())
print(f"<X0 X1> on {n_large} qubits = {large_estimate:.4f} ({time.perf_counter() - start:.2f} s for 1000 shots)")

<X0 X1> on 50 qubits = -0.0240 (0.28 s for 1000 shots)


## Running on a GPU

The sampler runs on the device of its choice. Passing a
`ProductBasisSampling(device="cuda")` instance as the `sampling_strategy` of the NIF device draws
the samples of every basis on the GPU, where large batches of shots are processed in parallel. Use
`dtype=torch.float32` for more speed on consumer GPUs, at the cost of precision on large circuits.
The cell below falls back to the CPU when no GPU is available.

In [10]:
compute_device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpu = 128
gpu_wires = list(range(n_gpu))


@qml.set_shots(10_000)
@qml.qnode(mc.NIFDevice(sampling_strategy=ProductBasisSampling(device=compute_device)))
def gpu_circuit():
    ProductState.from_basis_state([1, 0] * (n_gpu // 2), wires=gpu_wires)
    mc.operations.SingleParticleTransitionMatrixOperation.random(wires=gpu_wires, seed=2)
    return qml.expval(qml.X(0) @ qml.X(1))


gpu_circuit()  # warm-up
start = time.perf_counter()
gpu_estimate = float(gpu_circuit())
print(
    f"<X0 X1> on {n_gpu} qubits = {gpu_estimate:.4f} "
    f"({time.perf_counter() - start:.2f} s for 10000 shots on {compute_device})"
)

<X0 X1> on 128 qubits = 0.0898 (2.52 s for 10000 shots on cuda)


## Further reading

- The sampler is
  [`ProductBasisSampling`](https://github.com/MatchCake/MatchCake/blob/main/src/matchcake/devices/sampling_strategies/product_basis_sampling.py),
  whose docstring details the construction and its implementation on CPU and GPU. It is the default sampling strategy of the
  NIF device for product state inputs.
- D. J. Brod, *Efficient classical simulation of matchgate circuits with generalized inputs and
  measurements*, Phys. Rev. A **93**, 062332 (2016), [arXiv:1602.03539](https://arxiv.org/abs/1602.03539).
- The covariance matrix machinery shared with probabilities and expectation values is described
  in [`docs/expectation_values_theory.md`](../docs/expectation_values_theory.md).